# Lab 04-05 — Contextual Compression: shrink retrieved context before the LLM reads it

**Track 04 · Retrieval** — retrieval returns top-k *documents* — whole passages that are only partly about the question. Every irrelevant sentence is then paid for twice: once in context tokens, once in the answer LLM's attention. Contextual compression sits between the retriever and the answer step and cuts that waste.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and Groq directly — no repo component library. Every block of the pipeline is built right here: the local BGE embedder, the FAISS store, the base top-k retriever, the compressor LLM, and the `ContextualCompressionRetriever` that wraps them — which is exactly how the shared components in `src/` work underneath.

The idea, from LangChain's `ContextualCompressionRetriever`:

* a BASE RETRIEVER (here: FAISS over local BGE embeddings) returns the usual top-k passages;
* a COMPRESSOR — an LLM with an "extract the sentences relevant to this question" prompt (`LLMChainExtractor`) — reads each passage *and the question*, and returns only the query-relevant sentences, dropping the rest.

The compressor is called once per retrieved document, so the cost is `# questions x k` LLM round-trips (2 x 3 = 6 here, a couple of seconds each on Groq). That is the compression tax: you pay a small LLM bill to keep the *answer* LLM's context small.

This lab quantifies the payoff: for every question it prints the raw top-k context (characters + whitespace-token estimate) next to the compressed context and the reduction percentage.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-classic`, `langchain-community`, `langchain-huggingface`, `langchain-groq`, `sentence-transformers`, `faiss-cpu`, `python-dotenv`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script, and the repo-root `.env` (with `GROQ_API_KEY`) loads the same way. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS vector store (langchain_community)
#   langchain-classic     -> ContextualCompressionRetriever + LLMChainExtractor
#   langchain-groq        -> ChatGroq (the compressor LLM)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community langchain-classic langchain-groq python-dotenv pandas faiss-cpu


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# LangChain + sentence-transformers + faiss + Groq — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_classic.retrievers import (  # noqa: E402
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (  # noqa: E402
    LLMChainExtractor,
)
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
# (Gemini alternative: from langchain_google_genai import ChatGoogleGenerativeAI)

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes the deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610]` — 2 questions x `TOP_K = 3` = 6 compressor calls per run; `LLM_MODEL = "llama-3.3-70b-versatile"` is the Groq model doing the compression (a commented Gemini alternative is kept in the source). Groq is the *compressor* LLM, never the embedder — every embedding below is local BGE.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610]  # 2 questions x K=3 = 6 compressor calls per run
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *compressor* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` pulls the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line. Three tiny helpers round out the section: `tokens` is the cheap whitespace-split token estimate, and `total_chars` / `total_tokens` sum across a list of documents — the numbers the reduction percentage is computed from.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def tokens(text: str) -> int:
    """Cheap token estimate: whitespace-split word count."""
    return len(text.split())


def total_chars(docs: list[Document]) -> int:
    """Total characters across a list of documents."""
    return sum(len(d.page_content) for d in docs)


def total_tokens(docs: list[Document]) -> int:
    """Total token estimate across a list of documents."""
    return sum(tokens(d.page_content) for d in docs)


## 3. Experiment — raw retrieval vs LLM-compressed retrieval

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model; `FAISS.from_documents` builds the store in one call (embedding included, so its timing covers both). The base retriever is the store's `as_retriever(search_kwargs={"k": TOP_K})`; the compressor is `LLMChainExtractor.from_llm(ChatGroq(...))`; `ContextualCompressionRetriever` wraps the two. Every question runs through BOTH paths — raw and compressed — recording character counts, token estimates, and the compressor time. `run_experiment` returns a dict of artifacts instead of printing, so the demo and the gate read the same run.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — raw retrieval vs LLM-compressed retrieval; returns every
#    artifact the demo and the verification gate need (no re-computation
#    between the two paths)
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory with langchain-native FAISS -
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME, encode_kwargs={"normalize_embeddings": True}
    )
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedder)
    index_s = time.perf_counter() - t0

    # --- Base retriever (raw top-k) -----------------------------------------
    base_retriever = store.as_retriever(search_kwargs={"k": TOP_K})

    # --- Compressor LLM + the wrapped retriever ------------------------------
    # Groq only compresses; every embedding above is local BGE.
    # (Gemini alternative: llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.0))
    llm = ChatGroq(model=LLM_MODEL, temperature=0.0)
    compressor = LLMChainExtractor.from_llm(llm)
    compressed_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=base_retriever
    )

    # --- Per question: raw vs compressed -------------------------------------
    results = []
    for qid, qtext in questions:
        raw_docs = base_retriever.invoke(qtext)
        t0 = time.perf_counter()
        comp_docs = compressed_retriever.invoke(qtext)
        comp_s = time.perf_counter() - t0
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "raw_docs": raw_docs,
                "compressed_docs": comp_docs,
                "raw_chars": total_chars(raw_docs),
                "raw_tokens": total_tokens(raw_docs),
                "comp_chars": total_chars(comp_docs),
                "comp_tokens": total_tokens(comp_docs),
                "comp_s": comp_s,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "index_s": index_s,
        "results": results,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset and index timing; per question, the raw vs compressed context side by side (chars / token estimate), the reduction percentage and compressor time, and the raw top-1 / compressed first-hit previews; then a takeaway framing the compression tax — a small LLM bill (one call per retrieved document) for a smaller, cleaner answer context.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 05 — Contextual Compression: shrink context before the answer step")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} extractor")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Index:")
    print(f"    FAISS index built in {exp['index_s']:.3f}s over local BGE embeddings")

    print(f"\n[3] Raw vs compressed context (chars / token-estimate):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        raw_c, raw_t = r["raw_chars"], r["raw_tokens"]
        comp_c, comp_t = r["comp_chars"], r["comp_tokens"]
        print(f"      raw        {raw_c:5d} chars / {raw_t:4d} tokens  (k={len(r['raw_docs'])})")
        print(f"      compressed {comp_c:5d} chars / {comp_t:4d} tokens  (k={len(r['compressed_docs'])})")
        red_c = 100.0 * (raw_c - comp_c) / raw_c if raw_c else 0.0
        red_t = 100.0 * (raw_t - comp_t) / raw_t if raw_t else 0.0
        print(f"      reduction  {red_c:5.1f}% chars / {red_t:5.1f}% tokens "
              f"({r['comp_s']:.1f}s compressor time)")
        print("      raw top-1:  " + preview(r["raw_docs"][0].page_content))
        if r["compressed_docs"]:
            print("      compressed: " + preview(r["compressed_docs"][0].page_content))
        else:
            print("      compressed: (empty — extractor dropped every sentence)")

    print("\n[4] Takeaway")
    print("    The compressor keeps only the sentences the question is about,")
    print("    so the answer step reads a fraction of the original context.")
    print("    The price: one LLM call per retrieved document (2 questions x")
    print(f"    k={TOP_K} = {2 * TOP_K} calls here). Contextual compression trades")
    print("    that small LLM bill for a smaller, cleaner answer context —")
    print("    and the raw passages stay available if a question needs them.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages indexed; each question returns `TOP_K` raw hits; and per question — compressed context is non-empty, compressed chars strictly below raw chars (the extractor drops irrelevant sentences, so this holds with margin), and the compressed context still retains 'montevideo' (the answer's keyword must survive compression, not just shrink). The compression checks are pinned to properties that survive LLM wording variance. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))

    # Compression stability, pinned to properties that survive LLM wording
    # variance (the extractor keeps *which* sentences, not exact text).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"

        # The compressor must return at least one sentence.
        checks.append((f"{tag} compressed context is non-empty",
                       len(r["compressed_docs"]) > 0 and r["comp_chars"] > 0))

        # Compressed context must be strictly shorter than raw (in chars) —
        # LLMChainExtractor drops irrelevant sentences, so this holds with
        # margin; a small tolerance keeps it robust to 1-2 word overrides.
        checks.append((f"{tag} compressed chars < raw chars",
                       r["comp_chars"] < r["raw_chars"]))

        # The compressed context must still carry the answer's keyword —
        # the whole point is relevance, not just shrinkage. "Montevideo"
        # appears in the gold answer of both questions and survives the
        # extractor because the queries name it explicitly.
        joined = " ".join(d.page_content for d in r["compressed_docs"]).lower()
        checks.append((f"{tag} compressed context retains 'montevideo'",
                       "montevideo" in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of embedding + index build, then 6 Groq compressor calls (2 questions x k=3) — the live LLM calls happen here, once. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The raw vs compressed context side by side per question, with the reduction percentage and compressor time.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact or the compressor LLM call failed.


In [ ]:
verify_gate(exp)
